# Diffusion Planner: full architecture study

This notebook studies the complete released model architecture implemented in `src/diffusion_planner`. The default `PlannerConfig` is copied from the official released checkpoint `args.json`: 21 history frames, 80 future frames, 32 context agents, 10 predicted neighbors, 70 lanes, 25 route lanes, 192 hidden dimensions, six heads, and three encoder/decoder layers.

Every following section contains: the official module, a text data-flow diagram, the exact implementation source, and an input/output shape trace.

In [ ]:
import inspect
import torch
from diffusion_planner import DiffusionPlanner, PlannerConfig
from diffusion_planner.model import mixer, encoder, dit

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = PlannerConfig()
config

## 1. Full tensor contract

```text
neighbor_agents_past  (B, 32, 21, 11) -- AgentFusionEncoder --+
static_objects         (B,  5, 10)     -- StaticFusionEncoder -+--> context (B, 107, 192)
lanes                  (B, 70, 20, 12) -- LaneFusionEncoder --+
route_lanes            (B, 25, 20, 12) -- RouteEncoder -------> route condition (B, 192)
noisy trajectories     (B, 11, 324)    -- TrajectoryDiT ------> clean trajectories (B, 11, 324)
```

`324 = (80 future + 1 current) * 4`. The four state values are `x, y, cos(yaw), sin(yaw`. The first participant is ego; the remaining ten are neighbors.

In [ ]:
def make_inputs(batch=2):
    return {
        'ego_current_state': torch.randn(batch, 10, device=device),
        'neighbor_agents_past': torch.randn(batch, config.agent_num, config.time_len, 11, device=device),
        'static_objects': torch.randn(batch, config.static_objects_num, 10, device=device),
        'lanes': torch.randn(batch, config.lane_num, config.lane_len, 12, device=device),
        'lanes_speed_limit': torch.rand(batch, config.lane_num, 1, device=device),
        'lanes_has_speed_limit': torch.ones(batch, config.lane_num, 1, dtype=torch.bool, device=device),
        'route_lanes': torch.randn(batch, config.route_num, config.route_len, 12, device=device),
    }

inputs = make_inputs()
{name: tuple(value.shape) for name, value in inputs.items()}

## 2. MLP and MLP-Mixer

```text
vectors (B*elements, points, channels)
  -> LayerNorm -> transpose -> token MLP -> transpose -> residual
  -> LayerNorm -> channel MLP -> residual
```

The official source uses `timm.Mlp` and `timm.DropPath`; this implementation reproduces them locally so the architecture has no `timm` dependency.

In [ ]:
print(inspect.getsource(mixer.Mlp))
print(inspect.getsource(mixer.DropPath))
print(inspect.getsource(mixer.MixerBlock))
block = mixer.MixerBlock(20, 128, 0.1).to(device)
print('Mixer I/O:', (3, 20, 128), '->', tuple(block(torch.randn(3, 20, 128, device=device)).shape))

## 3. Agent-history encoder

```text
(B, 32, 21, 11) -> split state[8] and type[3]
state[8] + availability bit -> channel MLP (9 -> 128)
transpose -> time MLP (21 -> 64) -> Mixer blocks -> mean pool
+ type embedding (3 -> 128) -> MLP (128 -> 192)
```

The mask marks an all-zero agent history. The position token is the final history pose plus the agent modality one-hot `[1, 0, 0]`.

In [ ]:
print(inspect.getsource(encoder.AgentFusionEncoder))
agent_encoder = encoder.AgentFusionEncoder(config).to(device)
agent_tokens, agent_mask, agent_pos = agent_encoder(inputs['neighbor_agents_past'])
print('tokens:', tuple(agent_tokens.shape), 'mask:', tuple(agent_mask.shape), 'position:', tuple(agent_pos.shape))

## 4. Static-object encoder

```text
(B, 5, 10) -> MLP (10 -> 192) -> static tokens (B, 5, 192)
```

Static tokens are directly projected. Their position derives from pose and dimensions, then appends modality one-hot `[0, 1, 0]`.

In [ ]:
print(inspect.getsource(encoder.StaticFusionEncoder))
static_encoder = encoder.StaticFusionEncoder(config).to(device)
static_tokens, static_mask, static_pos = static_encoder(inputs['static_objects'])
print(tuple(static_tokens.shape), tuple(static_mask.shape), tuple(static_pos.shape))

## 5. Lane encoder

```text
(B, 70, 20, 12) -> geometry[8] -> channel MLP -> point MLP -> Mixer -> pool
speed limit (known or learned unknown embedding) + traffic light (4 -> 128)
-> MLP (128 -> 192)
```

This is a critical difference from a generic polyline encoder: speed limits and four-state traffic lights are separate additive embeddings, exactly as in the official model.

In [ ]:
print(inspect.getsource(encoder.LaneFusionEncoder))
lane_encoder = encoder.LaneFusionEncoder(config).to(device)
lane_tokens, lane_mask, lane_pos = lane_encoder(inputs['lanes'], inputs['lanes_speed_limit'], inputs['lanes_has_speed_limit'])
print(tuple(lane_tokens.shape), tuple(lane_mask.shape), tuple(lane_pos.shape))

## 6. Context fusion

```text
32 agent tokens + 5 static tokens + 70 lane tokens = 107 tokens
position projection (7 -> 192) + token
-> 3 x [LayerNorm -> masked self-attention -> residual -> MLP -> residual]
-> LayerNorm
```

Empty padded tokens are supplied as `key_padding_mask`. The first token is forcibly unmasked in the official code.

In [ ]:
print(inspect.getsource(encoder.SelfAttentionBlock))
print(inspect.getsource(encoder.FusionEncoder))
print(inspect.getsource(encoder.ContextEncoder))
context_encoder = encoder.ContextEncoder(config).to(device)
context = context_encoder(inputs)
{name: tuple(value.shape) for name, value in context.items()}

## 7. Route encoder and diffusion timestep

```text
route lanes (B, 25, 20, 12) -> take first 4 geometry features
flatten 25*20=500 points -> channel MLP (4 -> 64) -> point MLP (500 -> 32)
-> Mixer -> pool -> route token (B, 192)
time t (B,) -> sinusoid (256) -> MLP -> time token (B, 192)
condition = route token + time token
```

Navigation is therefore injected into every DiT block through the same condition as the diffusion noise level.

In [ ]:
print(inspect.getsource(dit.RouteEncoder))
print(inspect.getsource(dit.TimestepEmbedder))
route_encoder = dit.RouteEncoder(config).to(device)
time_encoder = dit.TimestepEmbedder(config.hidden_dim).to(device)
print('route:', tuple(route_encoder(inputs['route_lanes']).shape))
print('time:', tuple(time_encoder(torch.rand(2, device=device)).shape))

## 8. DiT block and final projection

```text
trajectory tokens -> adaLN self-attention -> gated residual
                  -> adaLN MLP            -> gated residual
                  -> cross-attention over 107 context tokens
                  -> MLP
                  -> adaLN final projection -> 324 values/token
```

The cross-attention and second MLP use the exact assignment structure of the public release. The initialization zeros adaLN modulation and output projection so the network begins from a stable diffusion baseline.

In [ ]:
print(inspect.getsource(dit.DiTBlock))
print(inspect.getsource(dit.FinalLayer))
print(inspect.getsource(dit.TrajectoryDiT))
denoiser = dit.TrajectoryDiT(config).to(device)
noisy = torch.randn(2, config.participant_num, config.trajectory_dim, device=device)
neighbor_mask = torch.zeros(2, config.predicted_neighbor_num, dtype=torch.bool, device=device)
print('denoiser:', tuple(denoiser(noisy, torch.rand(2, device=device), context['encoding'], inputs['route_lanes'], neighbor_mask).shape))

## 9. VP-SDE, current-state constraint, and released loss

```text
clean future -> state normalization -> VP noise at random t
prepend observed current state without noise
-> predict x_start for ego and every neighbor
-> MSE ego loss + MSE over valid neighbor futures
```

The current state is never diffused. This is the exact training construction used by the official `loss.py`; it is also the state constraint an inference solver must reapply after each denoising step.

In [ ]:
from diffusion_planner.diffusion import VPSchedule
print(inspect.getsource(VPSchedule))
print(inspect.getsource(DiffusionPlanner.diffusion_loss))
model = DiffusionPlanner(config).to(device).train()
ego_future = torch.randn(2, config.future_len, 4, device=device)
neighbor_future = torch.randn(2, config.predicted_neighbor_num, config.future_len, 4, device=device)
future_mask = torch.zeros(2, config.predicted_neighbor_num, config.future_len, dtype=torch.bool, device=device)
losses = model.diffusion_loss(inputs, ego_future, neighbor_future, future_mask)
losses

## 10. Inference boundary

The package exposes `model.denoise(x, t, encoded, inputs)` with the official DiT input/output contract for a DPM-Solver integrator. The official repository delegates integration to an upstream DPM-Solver implementation with 10 second-order multistep log-SNR steps. This notebook deliberately does not substitute a different sampler and label it as the official one.

The remaining non-architectural scope is nuPlan SDK preprocessing, distributed training, closed-loop simulation, and released-checkpoint weight compatibility. The architecture, initialization, structured encoder semantics, trajectory decoder, and training objective are implemented in this repository.